# 01 Build the slice cache (CPU, no GPU needed)
1. Session options: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN` attached.
2. **+ Add Input** -> your raw Dataset (`mnj-raw-ispy1`, `mnj-raw-ispy2` or `mnj-raw-duke`).
3. Set the parameters, then **Save Version -> Save & Run All (Commit)**.
4. When it finishes: check the QC numbers and the grid below, then **New Dataset** from the output,
   named `mnj-cache-<cohort>` (Private, shared with the team). Post the status counts in the chat.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "d1"          # code version; A posts the tag to use
PERSON = "B"            # "A", "B" or "C"
COHORT = "ispy1"       # "ispy1", "ispy2" or "duke"
LIMIT = 0               # 0 = all patients; 10 for a quick check
RAW = ""                # leave "" to find the mnj-raw-<cohort> input automatically

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
if not RAW:   # Kaggle may unpack the .tar.gz into a folder; either form works
    want = f"BreastDCEDL_{COHORT.upper()}_min_crop"
    hits = []
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if root.count("/") > 7:
            dirs[:] = []
            continue
        hits += [os.path.join(root, n) for n in dirs + files if n.startswith(want)]
        dirs[:] = [d for d in dirs if not d.startswith("BreastDCEDL_")]
    if not hits:
        for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
            print(root, dirs[:5], files[:5])
            if root.count("/") > 5:
                dirs[:] = []
    assert hits, f"no {want} folder or .tar.gz under /kaggle/input: attach mnj-raw-{COHORT} or set RAW"
    RAW = hits[0] if os.path.isdir(hits[0]) else os.path.dirname(hits[0])
OUT = f"/kaggle/working/cache_{COHORT}"
print("raw:", RAW, os.listdir(RAW))
cmd = [sys.executable, "scripts/build_cache.py", "--cohort", COHORT, "--raw", RAW, "--out", OUT, "--workers", "4"]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, display
idx = pd.read_csv(f"{OUT}/cache_index.csv")
print(json.load(open(f"{OUT}/cache_meta.json")))
print(idx.status.value_counts())
if "z_source" in idx:
    print(idx.z_source.value_counts(dropna=False))
if "xy_source" in idx:
    print(idx.xy_source.value_counts(dropna=False))
    sc = idx[idx.xy_source.isin(["mask", "csv_scaled"])]
    if "mask_csv_iou_yx" in sc and sc.mask_csv_iou_yx.notna().any():
        print("rescaled CSV box vs mask IoU:", sc.mask_csv_iou_yx.describe().round(3).to_dict())
if "mask_csv_iou_yx" in idx:
    print("mask vs CSV box IoU:", idx.mask_csv_iou_yx.describe().round(3).to_dict())
display(idx[idx.status == "error"].head(20))
display(Image(f"{OUT}/summary_grid.png"))